# This notebook converts the .mat data to netcdf format, 
# add the necessary metadata and plot the data

## load library

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import os, sys
import json

In [2]:
# Import modules
%reload_ext autoreload
%autoreload 2

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from metadata import create_json
from netcdf import read_mat_file, mat_to_xarray, save_to_netcdf

## INPUT REQUIRED: 
## Depth parameters from mooring diagram and mooring log 

In [3]:
instrument_height_above_bottom =  39   
water_depth_from_mooring_diagram = 4540
water_depth_from_ship_uncorrected = 4541     # uncorrected water depth 
water_depth_from_ship_corrected = 4541       # Replace with actual value, best water depth

#  Anchor (1) + chain (5) + nystron (20) + chain (5) + releases (1) + chain (5) 
# + 6 terminations at 0.25 ea (1.5) 
# + distance from termination on SBE cage to sensor (0.5) = 39 m

## read mat file

In [4]:
mat_filepath = '/Users/yugao/UOP/ORS-processing/data/external/stratus13/s13_sbe16_1875.mat'
# mat_filepath = '/Users/yugao/UOP/ORS-processing/data/external/stratus13/s13_sbe16_1873.mat'
mat_data = read_mat_file(mat_filepath)
# Inspect the structure of mat_data
# print(mat_data.keys()) 

In [5]:
mat_data

{'__header__': b'MATLAB 5.0 MAT-file, Platform: MACI64, Created on: Mon Apr 27 10:45:51 2015',
 '__version__': '1.0',
 '__globals__': [],
 'latitude': -19.624523333333332,
 'longitude': -84.95232333333334,
 'meta': array(('Stratus', 'Robert Weller', array(('surface mooring', 'Stratus Ocean Reference Station', 13, 2014, 4541, 3.7, 60, 'Ron Brown RB14-01', 'AGS Cabo de Hornos', '07-Mar-2014 18:01:01', '24-Apr-2015 11:34:01', array([735665.75070602, 736078.48195602]), 412.73124999995343),
       dtype=[('type', 'O'), ('experiment', 'O'), ('deployment_number', 'O'), ('start_year', 'O'), ('water_depth_m', 'O'), ('watch_circle_nm', 'O'), ('deck_height_cm', 'O'), ('deploymentcruise', 'O'), ('recoverycruise', 'O'), ('anchor_over_time', 'O'), ('anchor_release_time', 'O'), ('anchor_times', 'O'), ('duration', 'O')]), array(('WHOI', 'WHOI-UOP', 'Mooring observation', 'OceanSITES', 'Station', '38400'),
       dtype=[('institution', 'O'), ('data_assembly_center', 'O'), ('source', 'O'), ('naming_auth

## convert mat file to netcdf

In [6]:
sbe16_metadata = mat_data['meta']

In [8]:
import numpy as np
import xarray as xr

def extract_var_info(meta_data, var_name):
    # Attempt to safely extract calibration and unit information for a given variable
    try:
        for record in np.nditer(meta_data, flags=['refs_ok']):
            data = record.item()
            if var_name in data.dtype.names:
                var_info = data[var_name].item()
                units = var_info['units'].item() if 'units' in var_info.dtype.fields else 'unknown units'
                long_name = var_info['long_name'].item() if 'long_name' in var_info.dtype.fields else var_name.capitalize()
                cal_data = var_info['cal'].item() if 'cal' in var_info.dtype.fields else None
                return units, long_name, cal_data
    except Exception as e:
        print(f"Failed to extract metadata for {var_name}: {e}")
    return 'unknown units', var_name.capitalize(), None

def mat_to_xarray(mat_data):
    ds = xr.Dataset()
    
    # Handling time dimension
    time_dim = 'time'
    ds[time_dim] = xr.DataArray(data=np.ravel(mat_data['mday']), dims=[time_dim], attrs={'units': 'days since 1900-01-01'})

    # Handling variables
    meta_data = mat_data['meta'][0][0] if 'meta' in mat_data else None
    for var in ['temp', 'cond', 'sal', 'sal_sbe']:
        if var in mat_data:
            units, long_name, cal_data = extract_var_info(meta_data, var)
            data = np.ravel(mat_data[var])
            if cal_data:
                # Apply calibration based on the specifics of the cal_data structure
                G, H = float(cal_data['G'].item()), float(cal_data['H'].item())
                data = G * data + H  # Simple linear calibration for demonstration
            ds[var] = xr.DataArray(data=data, dims=[time_dim], attrs={'units': units, 'long_name': long_name})

    # Static attributes
    ds.attrs['depth'] = float(mat_data['depth'].item()) if 'depth' in mat_data else 'unknown'
    ds.attrs['latitude'] = float(mat_data['latitude'].item()) if 'latitude' in mat_data else 'unknown'
    ds.attrs['longitude'] = float(mat_data['longitude'].item()) if 'longitude' in mat_data else 'unknown'

    return ds

# Assuming mat_data is your loaded .mat file containing the data structure as described
ds_converted = mat_to_xarray(mat_data)
print(ds_converted)


IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [9]:
# Use the loaded MATLAB data (mat_data_updated) for conversion
ds = mat_to_xarray(mat_data)
ds

IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed

In [ ]:
ds.sal

In [ ]:
ds.sal_sbe

## Meta data: we will add depth paramter for deep T/S

In [ ]:
depth_parameters = {
    # consult mooring diagram
    'water_depth_from_mooring_diagram': water_depth_from_mooring_diagram,  
     # uncorrected water depth
    'water_depth_from_ship_uncorrected': water_depth_from_ship_uncorrected,
    # Replace with actual value, best water depth
    'water_depth_from_ship_corrected': water_depth_from_ship_corrected,        
    # instrument_depth_from_mooring_diagram = diagram depth  - height above bottom 
    'water_depth_from_pressure' : ds.attrs['depth'],
    'instrument_depth_from_mooring_diagram': water_depth_from_mooring_diagram - instrument_height_above_bottom,  
    # instrument_depth_from_mooring_log = corrected depth (4538.97 m) - height above bottom (39 m) 
    'instrument_depth_from_mooring_log': water_depth_from_ship_corrected - instrument_height_above_bottom,
    'instrument_height_above_bottom': 39
}

del ds.attrs['depth'] # delete depth 
depth_parameters 

In [ ]:
metadata_dict = create_json(mat_data, depth_parameters)

json_path = '/Users/yugao/UOP/ORS-processing/data/metadata/stratus_OS_NTAS_2016_D_TS.json'

# Convert the dictionary to a JSON string and save it
with open(json_path, 'w') as f:
    json.dump(metadata_dict, f)

### read json file

In [ ]:
with open(json_path, 'r') as f:
    metadata_from_json = json.load(f)

### incorporate the json file with all attributes

In [ ]:
metadata_from_json

In [ ]:
# Flatten the nested dictionaries within the 'attributes' attribute
attributes_flat = {}
for key, value in  metadata_from_json.items():
    if isinstance(value, dict):
        for sub_key, sub_value in value.items():
            attributes_flat[f'{sub_key}'] = sub_value
    else:
        attributes_flat[key] = value

ds.attrs.update(attributes_flat)

In [ ]:
netcdf_path = '/Users/yugao/UOP/ORS-processing/data/processed/stratus/stratus13' + mat_filepath[-15:-4] + '.nc'
netcdf_path 

In [ ]:
# Save the dataset to netCDF
ds.to_netcdf(netcdf_path)

In [ ]:
ds.attrs